# KAUST Kaggle Challenge 3 - Notebook B (FIXED)
## Offset-fault classification, strict forward-in-time validation

**Fix in this version:** submission configuration selection is now restricted to real model rows with a finite honestly-chosen threshold. Diagnostic rows such as `ordering_check` can no longer be selected as final models. Every submission also uses the threshold that belongs to its own feature/model configuration, with sanity checks against NaN thresholds and degenerate all-zero/all-one outputs.

The modelling and validation experiments are otherwise preserved from Notebook B.
### What the data actually says (established before any modelling)

Five things were checked on the real files first, and three of them contradict
the working plan we started from:

1. **Faults are isolated single rows, not episodes.** Median run length of the
   fault class is 1. The fault rate is flat at ~0.37 on *every* day and *every*
   hour. So "how long do fault runs last" and "does fault frequency drift" have
   no answer to find.
2. **Because faults are isolated, a row's temporal neighbours reveal it.** A
   faulty row sits far below its own local level on both sensors, and much
   further below on Sensor 2. This is the single largest source of signal.
3. **Test timestamps are minute-resolution; train timestamps are
   second-resolution.** Ordering train more precisely than we can ever order
   test silently inflates local validation. Both are floored to the minute here.
4. **The "2022" hint is not a free label.** January 2022 has the same
   fault-proxy share as December 2021 and as training. Calendar features are
   tested as a stage and they *hurt*, which is the defensible reading of hint 3.
5. **The S1-vs-S2 line is real but partial.** Best single-threshold rule on it
   reaches roughly F1 0.88 in train, short of the 0.922 published baseline.

Nothing below uses external data, external labels, or deep learning.

**This notebook's job:** never let a model see the future. Every fold trains on the past and validates on the days that follow, thresholds are chosen without peeking at the validation fold, and the timestamp-resolution trap is measured rather than assumed. Notebook A covers the date-grouped view.

## 0. Setup

In [ ]:
import os, sys, json, time, glob, warnings, gc
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupKFold

t_notebook_start = time.time()
SEED = 42
np.random.seed(SEED)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 80)
warnings.filterwarnings("ignore", category=FutureWarning)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
os.makedirs(WORK, exist_ok=True)

try:
    from catboost import CatBoostClassifier, Pool
    import catboost
    print("catboost", catboost.__version__)
except ImportError:
    print("catboost missing, installing...")
    os.system(f"{sys.executable} -m pip install -q catboost")
    from catboost import CatBoostClassifier, Pool

print("pandas", pd.__version__, "| numpy", np.__version__)
print("working dir:", WORK)

In [ ]:
# ---- dynamically locate the competition files -----------------------------
def find_files():
    roots = ["/kaggle/input", ".", "..", "/mnt/user-data/uploads"]
    found = {}
    for root in roots:
        if not os.path.isdir(root):
            continue
        for path in glob.glob(os.path.join(root, "**", "*.csv"), recursive=True):
            base = os.path.basename(path).lower()
            if base.startswith("train") and "train" not in found:
                found["train"] = path
            elif base.startswith("test") and "test" not in found:
                found["test"] = path
            elif "sample" in base and "sample" not in found:
                found["sample"] = path
        if "train" in found and "test" in found:
            break
    missing = {"train", "test"} - set(found)
    if missing:
        raise FileNotFoundError(f"could not locate {missing}. Saw: {found}")
    return found

FILES = find_files()
for k, v in FILES.items():
    print(f"{k:7s} -> {v}  ({os.path.getsize(v)/1e6:.1f} MB)")

In [ ]:
# ---- robust datetime parsing ---------------------------------------------
# train uses ISO with seconds; test uses M/D/YYYY H:MM with NO seconds.
# We never assume a single format.
def parse_datetime(series):
    s = series.astype(str)
    for fmt in ("%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%Y-%m-%d %H:%M"):
        out = pd.to_datetime(s, format=fmt, errors="coerce")
        if out.notna().all():
            return out, fmt
    out = pd.to_datetime(s, errors="coerce", format="mixed", dayfirst=False)
    if out.isna().any():
        raise ValueError(f"{out.isna().sum()} timestamps failed to parse")
    return out, "mixed"

train = pd.read_csv(FILES["train"])
test  = pd.read_csv(FILES["test"])
sample = pd.read_csv(FILES["sample"]) if "sample" in FILES else None

train["dt"], fmt_tr = parse_datetime(train["Datetime"])
test["dt"],  fmt_te = parse_datetime(test["Datetime"])
print("train datetime format:", fmt_tr, "| test datetime format:", fmt_te)

TARGET = "Offset_fault"
S1, S2 = "Sensor1_PM2.5", "Sensor2_PM2.5"
EPS = 1e-3

print("\ntrain", train.shape, train["dt"].min(), "->", train["dt"].max())
print("test ", test.shape,  test["dt"].min(),  "->", test["dt"].max())

# sanity: month-first parsing must place every test row inside Dec21-Jan22
assert test["dt"].between(pd.Timestamp("2021-11-30"), pd.Timestamp("2022-02-01")).all(), \
    "test datetimes parsed outside the expected window - check day/month order"
assert train["dt"].max() < test["dt"].min(), "train/test are not chronologically disjoint"
print("\nOK: parsing validated, train ends before test begins (pure forecasting split).")

## 1. EDA that tries to break our assumptions

In [ ]:
# ---- EDA 1: target balance, missingness -----------------------------------
print("target balance")
print(train[TARGET].value_counts(normalize=True).round(4).to_string(), "\n")

print("missing values (train)")
print(train.isna().sum()[lambda s: s > 0].to_string())
print("\nmissing values (test)")
print(test.isna().sum()[lambda s: s > 0].to_string())

na_sensor_tr = train[S1].isna()
print(f"\nrows with BOTH sensors missing: train={int((train[S1].isna() & train[S2].isna()).sum())}, "
      f"test={int((test[S1].isna() & test[S2].isna()).sum())}")
print(f"fault rate | sensors missing : {train.loc[na_sensor_tr, TARGET].mean():.4f}")
print(f"fault rate | temp missing    : {train.loc[train['Temperature'].isna(), TARGET].mean():.4f}")
print(f"fault rate | overall         : {train[TARGET].mean():.4f}")
print("\nNOTE: test has ZERO rows with missing sensors, train has thousands. Those train rows")
print("carry no usable sensor signal and represent a regime the test set never contains.")

In [ ]:
# ---- EDA 2: sampling structure and the timestamp-resolution trap ----------
tr_sorted = train.sort_values("dt")
gaps = tr_sorted["dt"].diff().dt.total_seconds().dropna()
print("train inter-row gap (seconds)")
print(gaps.describe(percentiles=[.1, .25, .5, .75, .9, .99]).round(2).to_string())

print(f"\nfraction of timestamps with seconds == 0:  train={(train['dt'].dt.second == 0).mean():.4f}"
      f"  test={(test['dt'].dt.second == 0).mean():.4f}")

tr_min = train["dt"].dt.floor("min")
te_min = test["dt"].dt.floor("min")
print(f"\nrows per distinct minute: train={len(train)/tr_min.nunique():.3f}  test={len(test)/te_min.nunique():.3f}")
print("rows-per-minute histogram (train):", tr_min.value_counts().value_counts().sort_index().to_dict())
print("rows-per-minute histogram (test) :", te_min.value_counts().value_counts().sort_index().to_dict())

print("""
=========================== CRITICAL FINDING ===========================
The TEST timestamps are truncated to the minute (every second == 0).
The TRAIN timestamps keep full second resolution.

Both files are the SAME underlying stream sampled ~2.18 rows per minute.

Any feature that depends on row ORDER (lags, neighbours, rolling windows)
can therefore be built more accurately on train than it ever can on test.
If we sort train by its exact second and test by minute, local validation
is measuring a precision we will not have at inference time.

=> Every notebook below floors BOTH tables to the minute and then breaks
   within-minute ties with the same arbitrary rule. Notebook B quantifies
   exactly how much this handicap costs.
========================================================================
""")

In [ ]:
# ---- EDA 3: are faults temporally clustered? ------------------------------
y_seq = tr_sorted[TARGET].values
change = np.flatnonzero(np.diff(y_seq)) + 1
run_lens = np.diff(np.concatenate([[0], change, [len(y_seq)]]))
run_lab = y_seq[np.concatenate([[0], change])]
runs = pd.DataFrame({"label": run_lab, "len": run_lens})
print(f"number of label runs: {len(runs):,} across {len(y_seq):,} rows")
print(runs.groupby("label")["len"].describe(percentiles=[.5, .9, .99]).round(2).to_string())

daily = train.assign(date=train["dt"].dt.date).groupby("date").agg(
    n=(TARGET, "size"), fault_rate=(TARGET, "mean"))
print(f"\ndaily fault rate: min={daily.fault_rate.min():.4f} max={daily.fault_rate.max():.4f} "
      f"std={daily.fault_rate.std():.5f} over {len(daily)} days")

hourly = train.assign(h=train["dt"].dt.hour).groupby("h")[TARGET].mean()
print(f"hourly fault rate: min={hourly.min():.4f} max={hourly.max():.4f} std={hourly.std():.5f}")

print("""
FINDING (contradicts the working plan):
Faults are NOT emitted in contiguous episodes. Median run length is 1 row
for the fault class. Faults are injected into individual, isolated rows.
The fault rate is also essentially constant across every day and every hour.

Two consequences:
  * "how long does a fault episode last" and "does fault frequency drift"
    are dead ends - there is nothing there to model.
  * BUT because a faulty row sits between healthy neighbours, its temporal
    neighbours give a clean estimate of the true ambient level at that
    moment. That is the exploitable structure.
""")

In [ ]:
# ---- EDA 4: hint 1, "you just need that line" -----------------------------
d = train.dropna(subset=[S1, S2]).copy()
d["diff"]  = d[S1] - d[S2]
d["ratio"] = d[S1] / (d[S2].abs() + EPS)
d["rel"]   = d["diff"] / (d[S2].abs() + EPS)

for lab in (0, 1):
    s = d[d[TARGET] == lab]
    slope, intercept = np.polyfit(s[S2], s[S1], 1)
    r = np.corrcoef(s[S2], s[S1])[0, 1]
    print(f"class {lab}: S1 = {slope:.4f} * S2 + {intercept:.4f}   (corr {r:.4f}, n={len(s):,})")

print()
print(d.groupby(TARGET)[["diff", "ratio", "rel"]]
        .describe(percentiles=[.01, .25, .5, .75, .99]).round(3).T.to_string())

# how far does the line alone get us?
best = (0.0, None, None)
for col in ("diff", "ratio", "rel"):
    for thr in np.quantile(d[col], np.linspace(0.02, 0.98, 300)):
        f = f1_score(d[TARGET], (d[col] >= thr).astype(int))
        if f > best[0]:
            best = (f, col, thr)
print(f"\nbest single-feature threshold rule: {best[1]} >= {best[2]:.4f}  ->  train F1 = {best[0]:.4f}")
print("""
The line is real: healthy rows sit on S1 ~= S2, faulty rows sit on a
visibly steeper line. But a threshold on it alone tops out well short of
the published 92.2% leaderboard baseline, so the line is a component of
the answer, not the answer.
""")

In [ ]:
# ---- EDA 5: the local-context signal --------------------------------------
# Rolling median over neighbouring rows estimates the ambient level.
# No labels are used, so this is computable identically on the test set.
probe = train.sort_values("dt").reset_index(drop=True)
for w in (7, 21, 121):
    for col, nm in ((S1, "S1"), (S2, "S2")):
        med = probe[col].rolling(w, center=True, min_periods=3).median()
        probe[f"{nm}_localratio_{w}"] = probe[col] / (med.abs() + EPS)

cols = [c for c in probe.columns if "localratio" in c]
summary = probe.groupby(TARGET)[cols].median().T
summary.columns = ["healthy", "faulty"]
summary["separation"] = (summary["healthy"] - summary["faulty"]).abs()
print("median of value / local-median, by class")
print(summary.round(4).to_string())

print("""
FINDING: a faulty row sits far BELOW its own temporal neighbourhood on both
sensors, and much further below on Sensor 2 than on Sensor 1. That is the
whole fault mechanism visible in one table, and it is exactly what hints 4
(lags) and 5 (aggregate) are pointing at. It also explains why the fault
class shows a steeper S1-vs-S2 line: S2 is pulled down harder than S1.
""")

In [ ]:
# ---- EDA 6: train/test drift and the "2022" hypothesis --------------------
def reldiff(df):
    return (df[S1] - df[S2]) / (df[S2].abs() + EPS)

train["_rel"] = reldiff(train)
test["_rel"]  = reldiff(test)

comp = pd.DataFrame({
    "train": train[[S1, S2, "Temperature", "Relative_Humidity", "_rel"]].median(),
    "test":  test[[S1, S2, "Temperature", "Relative_Humidity", "_rel"]].median(),
}).round(3)
print("median of each column, train vs test")
print(comp.to_string())

# A crude fault-share proxy that needs no labels: share of rows above a
# separating threshold on the relative difference.
THR_PROXY = 0.15
print(f"\nshare of rows with rel-diff > {THR_PROXY} (proxy for fault share)")
print(f"  train  : {(train['_rel'] > THR_PROXY).mean():.4f}   (true fault rate {train[TARGET].mean():.4f})")
by_month = test.assign(m=test["dt"].dt.to_period("M")).groupby("m")["_rel"].agg(
    n="size", proxy=lambda s: (s > THR_PROXY).mean()).round(4)
print(by_month.to_string())

print("""
FINDING on hint 3 ("I am 2022, I promise nothing wrong will happen"):
January 2022 shows the SAME fault-proxy share as December 2021 and as the
training period. There is no evidence that 2022 rows are disproportionately
healthy, and a hard rule mapping 2022 -> class 0 would destroy roughly half
the positive class on the private leaderboard.

The defensible reading is the boring one: train contains a single year, so
any raw year / month / day-of-year feature is a constant in training and an
unseen value at inference. The stage-by-stage runs below test this directly,
and calendar features do measurably HURT on future validation.

Temperature and relative humidity also drift (autumn -> winter), so they are
the covariates most likely to carry a spurious seasonal association. An
ablation below checks whether the model actually needs them.
""")

## 2. Feature construction

In [ ]:
# ---- unified stream + feature builder -------------------------------------
# Features are built on train and test CONCATENATED, sorted by time. No label
# is ever consulted, no external data is used: this is only a way to give a
# test row the same kind of temporal neighbourhood a train row gets.

ORDER_SEED = 0          # controls arbitrary within-minute tie-breaking
K_NEIGHBOURS = 3        # neighbours used on each side (tuned in stage 5)

def build_stream(order_seed=ORDER_SEED):
    a = train.drop(columns=[c for c in ["_rel"] if c in train.columns]).copy()
    b = test.drop(columns=[c for c in ["_rel"] if c in test.columns]).copy()
    a["ID"] = -1
    a["is_test"] = 0
    b["is_test"] = 1
    b[TARGET] = np.nan
    full = pd.concat([a, b], ignore_index=True, sort=False)

    # Floor to the minute so train gets exactly the resolution test has.
    full["tmin"] = full["dt"].dt.floor("min")
    rng = np.random.RandomState(order_seed)
    full["_tie"] = rng.rand(len(full))
    full = full.sort_values(["tmin", "_tie"]).reset_index(drop=True)
    return full

def add_raw(f, full):
    f["S1"] = full[S1]
    f["S2"] = full[S2]
    f["Temperature"] = full["Temperature"]
    f["Relative_Humidity"] = full["Relative_Humidity"]
    return ["S1", "S2", "Temperature", "Relative_Humidity"]

def add_relation(f, full):
    diff = full[S1] - full[S2]
    f["diff"] = diff
    f["abs_diff"] = diff.abs()
    f["ratio"] = full[S1] / (full[S2].abs() + EPS)
    f["rel_diff"] = diff / (full[S2].abs() + EPS)
    f["sym_rel"] = 2 * diff / (full[S1].abs() + full[S2].abs() + EPS)
    f["log_ratio"] = np.log(full[S1].clip(lower=0.01)) - np.log(full[S2].clip(lower=0.01))
    f["level"] = (full[S1] + full[S2]) / 2
    f["resid_line"] = full[S1] - (1.0039 * full[S2] + 0.7551)   # healthy-class fit from EDA 4
    return ["diff", "abs_diff", "ratio", "rel_diff", "sym_rel", "log_ratio", "level", "resid_line"]

def add_calendar(f, full):
    f["hour"] = full["dt"].dt.hour
    f["dayofweek"] = full["dt"].dt.dayofweek
    f["day"] = full["dt"].dt.day
    f["minute_of_day"] = full["dt"].dt.hour * 60 + full["dt"].dt.minute
    f["hour_sin"] = np.sin(2 * np.pi * f["minute_of_day"] / 1440)
    f["hour_cos"] = np.cos(2 * np.pi * f["minute_of_day"] / 1440)
    return ["hour", "dayofweek", "day", "minute_of_day", "hour_sin", "hour_cos"]

def add_neighbour(f, full, k=K_NEIGHBOURS):
    """Lag/lead context. Faults are isolated rows, so neighbours estimate ambient."""
    names = []
    ratio = full[S1] / (full[S2].abs() + EPS)
    for col, nm in ((S1, "S1"), (S2, "S2"), (None, "ratio")):
        base = ratio if col is None else full[col]
        sides = {}
        for j in range(1, k + 1):
            sides[f"m{j}"] = base.shift(j)
            sides[f"p{j}"] = base.shift(-j)
            if col is not None:
                f[f"{nm}_lag{j}"] = sides[f"m{j}"]
                f[f"{nm}_lead{j}"] = sides[f"p{j}"]
                names += [f"{nm}_lag{j}", f"{nm}_lead{j}"]
        N = pd.DataFrame(sides)
        med, mean, sd = N.median(axis=1), N.mean(axis=1), N.std(axis=1)
        f[f"{nm}_nb_med"] = med
        f[f"{nm}_nb_std"] = sd
        f[f"{nm}_nb_dev"] = base - med
        f[f"{nm}_nb_ratio"] = base / (med.abs() + EPS)
        f[f"{nm}_nb_z"] = (base - mean) / (sd + EPS)
        names += [f"{nm}_nb_med", f"{nm}_nb_std", f"{nm}_nb_dev", f"{nm}_nb_ratio", f"{nm}_nb_z"]

    f["nb_ratio_gap"] = f["S1_nb_ratio"] / (f["S2_nb_ratio"].abs() + EPS)
    f["nb_dev_gap"] = f["S1_nb_dev"] - f["S2_nb_dev"]
    f["gap_prev_s"] = (full["tmin"] - full["tmin"].shift(1)).dt.total_seconds()
    f["gap_next_s"] = (full["tmin"].shift(-1) - full["tmin"]).dt.total_seconds()
    names += ["nb_ratio_gap", "nb_dev_gap", "gap_prev_s", "gap_next_s"]
    return names

def add_rolling(f, full, windows=("30min", "360min")):
    """Time-based windows: immune to within-minute ordering, unlike row windows."""
    names = []
    g = full.set_index("tmin")
    ratio_vals = (full[S1] / (full[S2].abs() + EPS)).values
    for w in windows:
        for col, nm in ((S1, "S1"), (S2, "S2")):
            med = g[col].rolling(w, center=True, closed="both").median().values
            f[f"{nm}_roll_ratio_{w}"] = full[col].values / (np.abs(med) + EPS)
            f[f"{nm}_roll_dev_{w}"] = full[col].values - med
            names += [f"{nm}_roll_ratio_{w}", f"{nm}_roll_dev_{w}"]
        rmed = (g[S1] / (g[S2].abs() + EPS)).rolling(w, center=True, closed="both").median().values
        f[f"ratio_roll_dev_{w}"] = ratio_vals - rmed
        names.append(f"ratio_roll_dev_{w}")
    return names

def build_features(order_seed=ORDER_SEED, k=K_NEIGHBOURS):
    full = build_stream(order_seed)
    f = pd.DataFrame(index=full.index)
    groups = {}
    groups["raw"] = add_raw(f, full)
    groups["relation"] = add_relation(f, full)
    groups["calendar"] = add_calendar(f, full)
    groups["neighbour"] = add_neighbour(f, full, k)
    groups["rolling"] = add_rolling(f, full)
    f = f.replace([np.inf, -np.inf], np.nan).astype(np.float32)
    return full, f, groups

t0 = time.time()
full, FEAT, GROUPS = build_features()
print(f"feature matrix {FEAT.shape} built in {time.time()-t0:.1f}s")
for g, cols in GROUPS.items():
    print(f"  {g:10s} {len(cols):3d} cols")

is_train = (full["is_test"] == 0).values
X_all = FEAT[is_train].reset_index(drop=True)
y_all = full.loc[is_train, TARGET].astype(int).values
dt_all = full.loc[is_train, "dt"].reset_index(drop=True)
X_test = FEAT[~is_train].reset_index(drop=True)
test_ids = full.loc[~is_train, "ID"].astype(int).values
print(f"\nX_all {X_all.shape} | X_test {X_test.shape} | positive rate {y_all.mean():.4f}")

STAGES = {
    "S0_raw":            GROUPS["raw"],
    "S1_relation":       GROUPS["raw"] + GROUPS["relation"],
    "S2_calendar":       GROUPS["raw"] + GROUPS["relation"] + GROUPS["calendar"],
    "S3_neighbour":      GROUPS["raw"] + GROUPS["relation"] + GROUPS["neighbour"],
    "S4_rolling":        GROUPS["raw"] + GROUPS["relation"] + GROUPS["neighbour"] + GROUPS["rolling"],
}

## 3. Validation framework

Model settings are frozen across stages 0-4 so that every score change is
attributable to the feature family, not to tuning.

The threshold is chosen on an *inner* slice that ends before the holdout begins, so the reported holdout F1 is achievable in practice. The oracle threshold is printed alongside purely to show how much is being left on the table.

In [ ]:
# ---- fixed model settings + threshold optimiser + metric logger -----------
CB_PARAMS = dict(
    iterations=600,
    learning_rate=0.08,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="Logloss",
    random_seed=SEED,
    thread_count=-1,
    verbose=0,
    allow_writing_files=False,
)
print("frozen CatBoost params for all feature stages:", CB_PARAMS)

def optimise_threshold(y_true, proba, n_grid=300):
    """Pick the probability cut that maximises F1 on the validation fold."""
    lo, hi = np.quantile(proba, 0.01), np.quantile(proba, 0.99)
    grid = np.unique(np.concatenate([np.linspace(lo, hi, n_grid), [0.5]]))
    scores = [f1_score(y_true, (proba >= t).astype(int)) for t in grid]
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])

def fold_report(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return dict(f1=f, precision=p, recall=r, threshold=threshold,
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),
                pos_rate=float(pred.mean()), auc=float(roc_auc_score(y_true, proba)))

METRIC_LOG = []

def log_result(row):
    METRIC_LOG.append(row)
    return row

def save_log(name):
    df = pd.DataFrame(METRIC_LOG)
    path = os.path.join(WORK, name)
    df.to_csv(path, index=False)
    print(f"saved {path}  ({len(df)} rows)")
    return df

def fit_predict(Xtr, ytr, Xva, params=None):
    m = CatBoostClassifier(**(params or CB_PARAMS))
    m.fit(Xtr, ytr)
    return m, m.predict_proba(Xva)[:, 1]

In [ ]:
# ---- validation B: forward chaining + a strict future holdout -------------
# Every fold trains only on the past and validates on the immediate future,
# which is the exact shape of the real train -> test problem. The last fold
# (late November) is the closest available analogue to December.

FOLD_CUTS = [
    ("2021-10-15", "2021-11-02", "2021-11-09"),
    ("2021-10-15", "2021-11-09", "2021-11-16"),
    ("2021-10-15", "2021-11-16", "2021-11-23"),
    ("2021-10-15", "2021-11-23", "2021-12-01"),
]
HOLDOUT_START = "2021-11-24"

def run_temporal_cv(cols, stage_name, params=None, verbose=True):
    t0 = time.time()
    fold_rows = []
    oof = np.full(len(X_all), np.nan)
    for k, (s, mid, e) in enumerate(FOLD_CUTS):
        itr = np.flatnonzero((dt_all >= pd.Timestamp(s)) & (dt_all < pd.Timestamp(mid)))
        iva = np.flatnonzero((dt_all >= pd.Timestamp(mid)) & (dt_all < pd.Timestamp(e)))
        _, proba = fit_predict(X_all.loc[itr, cols], y_all[itr], X_all.loc[iva, cols], params)
        oof[iva] = proba
        thr, _ = optimise_threshold(y_all[iva], proba)
        rep = fold_report(y_all[iva], proba, thr)
        rep.update(fold=k, n_train=len(itr), n_val=len(iva), val_span=f"{mid[5:]}..{e[5:]}")
        fold_rows.append(rep)
        if verbose:
            print(f"  fold {k} (val {rep['val_span']}, n={len(iva):,}): F1={rep['f1']:.5f} "
                  f"P={rep['precision']:.4f} R={rep['recall']:.4f} thr={thr:.3f} "
                  f"posrate={rep['pos_rate']:.4f}")

    folds = pd.DataFrame(fold_rows)

    # strict single holdout, threshold chosen on earlier data only
    i_pre = np.flatnonzero(dt_all < pd.Timestamp(HOLDOUT_START))
    i_hold = np.flatnonzero(dt_all >= pd.Timestamp(HOLDOUT_START))
    inner_cut = pd.Timestamp(HOLDOUT_START) - pd.Timedelta(days=7)
    i_in_tr = np.flatnonzero(dt_all < inner_cut)
    i_in_va = np.flatnonzero((dt_all >= inner_cut) & (dt_all < pd.Timestamp(HOLDOUT_START)))
    _, p_in = fit_predict(X_all.loc[i_in_tr, cols], y_all[i_in_tr], X_all.loc[i_in_va, cols], params)
    thr_honest, _ = optimise_threshold(y_all[i_in_va], p_in)

    _, p_hold = fit_predict(X_all.loc[i_pre, cols], y_all[i_pre], X_all.loc[i_hold, cols], params)
    hold_honest = fold_report(y_all[i_hold], p_hold, thr_honest)
    thr_oracle, f_oracle = optimise_threshold(y_all[i_hold], p_hold)

    row = dict(stage=stage_name, n_features=len(cols),
               fwd_f1_mean=folds.f1.mean(), fwd_f1_std=folds.f1.std(),
               fwd_f1_last=folds.f1.iloc[-1], thr_mean=folds.threshold.mean(),
               holdout_f1_honest_thr=hold_honest["f1"], holdout_thr_honest=thr_honest,
               holdout_f1_oracle_thr=f_oracle, holdout_thr_oracle=thr_oracle,
               holdout_precision=hold_honest["precision"], holdout_recall=hold_honest["recall"],
               holdout_pos_rate=hold_honest["pos_rate"], holdout_auc=hold_honest["auc"],
               holdout_f1_at_half=f1_score(y_all[i_hold], (p_hold >= 0.5).astype(int)),
               tn=hold_honest["tn"], fp=hold_honest["fp"], fn=hold_honest["fn"], tp=hold_honest["tp"],
               runtime_s=round(time.time() - t0, 1))
    log_result(row)
    print(f"  -> {stage_name}: forward F1 {row['fwd_f1_mean']:.5f} +/- {row['fwd_f1_std']:.5f} | "
          f"holdout F1 {row['holdout_f1_honest_thr']:.5f} (honest thr {thr_honest:.3f}, "
          f"oracle would give {f_oracle:.5f}) | {row['runtime_s']}s")
    np.save(os.path.join(WORK, f"holdout_proba_{stage_name}.npy"), p_hold)
    return row, folds

## 4. Staged experiments

In [ ]:
# ---- stages 0-4 under strict temporal validation --------------------------
fold_store = {}
for stage_name, cols in STAGES.items():
    print(f"\n=== {stage_name}  ({len(cols)} features) ===")
    row, folds = run_temporal_cv(cols, stage_name)
    fold_store[stage_name] = folds

results = pd.DataFrame(METRIC_LOG)
print("\n" + "=" * 110)
print(results[["stage", "n_features", "fwd_f1_mean", "fwd_f1_std", "fwd_f1_last",
               "holdout_f1_honest_thr", "holdout_thr_honest", "holdout_f1_oracle_thr",
               "holdout_precision", "holdout_recall", "runtime_s"]].round(5).to_string(index=False))
print("""
The gap between holdout_f1_honest_thr and holdout_f1_oracle_thr is the price of
having to choose the threshold in advance. If it is small, threshold choice is
not a risk worth spending submissions on.
""")

## 5. The timestamp-resolution trap, measured

In [ ]:
# ---- the timestamp-resolution trap, measured ------------------------------
# Variant 1 (honest): both tables floored to the minute, ties broken randomly.
# Variant 2 (optimistic): train sorted by its exact second, which test can
#                         never be. This is the number we would have believed.
def build_features_exact_sort(k=K_NEIGHBOURS):
    a = train.drop(columns=[c for c in ["_rel"] if c in train.columns]).copy()
    b = test.drop(columns=[c for c in ["_rel"] if c in test.columns]).copy()
    a["ID"] = -1; a["is_test"] = 0
    b["is_test"] = 1; b[TARGET] = np.nan
    fl = pd.concat([a, b], ignore_index=True, sort=False)
    fl["tmin"] = fl["dt"].dt.floor("min")
    fl = fl.sort_values("dt").reset_index(drop=True)     # exact seconds where available
    f = pd.DataFrame(index=fl.index)
    g = {}
    g["raw"] = add_raw(f, fl); g["relation"] = add_relation(f, fl)
    g["calendar"] = add_calendar(f, fl); g["neighbour"] = add_neighbour(f, fl, k)
    g["rolling"] = add_rolling(f, fl)
    return fl, f.replace([np.inf, -np.inf], np.nan).astype(np.float32), g

cols_best = STAGES["S4_rolling"]

def holdout_only(Xa, ya, dta, cols, label):
    i_pre = np.flatnonzero(dta < pd.Timestamp(HOLDOUT_START))
    i_hold = np.flatnonzero(dta >= pd.Timestamp(HOLDOUT_START))
    _, p = fit_predict(Xa.loc[i_pre, cols], ya[i_pre], Xa.loc[i_hold, cols])
    thr, f = optimise_threshold(ya[i_hold], p)
    print(f"  {label:34s} holdout F1 @0.5 = {f1_score(ya[i_hold], (p>=0.5).astype(int)):.5f}   best = {f:.5f}")
    return f

print("effect of timestamp resolution on measured performance")
f_honest = holdout_only(X_all, y_all, dt_all, cols_best, "minute resolution (honest)")
fl_e, feat_e, grp_e = build_features_exact_sort()
m_e = (fl_e["is_test"] == 0).values
f_opt = holdout_only(feat_e[m_e].reset_index(drop=True),
                     fl_e.loc[m_e, TARGET].astype(int).values,
                     fl_e.loc[m_e, "dt"].reset_index(drop=True),
                     cols_best, "exact seconds (unachievable)")
print(f"\n  optimism if we had ignored this: {f_opt - f_honest:+.5f} F1")
del fl_e, feat_e; gc.collect()

# how much does the arbitrary within-minute order itself move the score?
print("\nsensitivity to the arbitrary within-minute ordering")
seed_scores = []
for s in (0, 1, 2):
    fl_s, feat_s, grp_s = build_features(order_seed=s)
    m_s = (fl_s["is_test"] == 0).values
    f_s = holdout_only(feat_s[m_s].reset_index(drop=True),
                       fl_s.loc[m_s, TARGET].astype(int).values,
                       fl_s.loc[m_s, "dt"].reset_index(drop=True),
                       cols_best, f"order_seed={s}")
    seed_scores.append(f_s)
    del fl_s, feat_s; gc.collect()
print(f"  spread across orderings: {np.std(seed_scores):.5f} "
      f"(min {min(seed_scores):.5f}, max {max(seed_scores):.5f})")
log_result(dict(stage="ordering_check", holdout_f1_honest_thr=f_honest,
                holdout_f1_oracle_thr=f_opt, fwd_f1_std=float(np.std(seed_scores))))

## 6. Hint 3 on trial

In [ ]:
# ---- hint 3 put on trial: does "2022" mean anything? ----------------------
# Training contains only 2021, so this cannot be tested directly. What we CAN
# do is (a) check whether the unlabelled test months differ from each other and
# from train, and (b) measure what a 2022 -> class 0 rule would cost if the
# same rule were applied to an equivalent slice of labelled training data.

proxy_thr = 0.15
rel_tr = (train[S1] - train[S2]) / (train[S2].abs() + EPS)
rel_te = (test[S1] - test[S2]) / (test[S2].abs() + EPS)
print("fault-proxy share (rel-diff above threshold)")
print(f"  train (true fault rate {train[TARGET].mean():.4f}) : {(rel_tr > proxy_thr).mean():.4f}")
for period, mask in [("test Dec 2021", test['dt'].dt.year == 2021),
                     ("test Jan 2022", test['dt'].dt.year == 2022)]:
    print(f"  {period:16s}: {(rel_te[mask] > proxy_thr).mean():.4f}  (n={int(mask.sum()):,})")

# cost of blanket-zeroing a comparable share of rows
share_2022 = float((test["dt"].dt.year == 2022).mean())
order = np.argsort(dt_all.values)
n_zero = int(share_2022 * len(y_all))
tail = order[-n_zero:]
y_pred_perfect = y_all.copy()
y_pred_zeroed = y_all.copy()
y_pred_zeroed[tail] = 0
print(f"\nif the last {share_2022:.1%} of the training period had been forced to class 0,")
print(f"a model that was otherwise PERFECT would score F1 = {f1_score(y_all, y_pred_zeroed):.5f} "
      f"instead of {f1_score(y_all, y_pred_perfect):.5f}")
print("""
VERDICT: there is no evidence for the literal reading, and the downside is
catastrophic on a 90% private split. The rule is rejected. Hint 3 is read as a
warning that year/month/day features are constant in training and unseen at
inference - which the S2_calendar stage above already demonstrates.
""")

## 7. Ablations

In [ ]:
# ---- ablations under strict temporal validation ---------------------------
best_cols = STAGES["S4_rolling"]
ablations = {
    "B_full":           best_cols,
    "B_no_weather":     [c for c in best_cols if c not in ("Temperature", "Relative_Humidity")],
    "B_no_nan_rows":    best_cols,      # handled below via a training mask
    "B_plus_calendar":  best_cols + GROUPS["calendar"],
}
for name, cols in ablations.items():
    if name == "B_no_nan_rows":
        continue
    print(f"\n=== {name} ({len(cols)} features) ===")
    run_temporal_cv(cols, name, verbose=False)

# test never contains a row with both sensors missing; train does. Do those
# rows help or just add noise?
sensor_na = X_all["S1"].isna().values
i_pre = np.flatnonzero((dt_all < pd.Timestamp(HOLDOUT_START)))
i_hold = np.flatnonzero(dt_all >= pd.Timestamp(HOLDOUT_START))
for drop_na in (False, True):
    itr = i_pre[~sensor_na[i_pre]] if drop_na else i_pre
    _, p = fit_predict(X_all.loc[itr, best_cols], y_all[itr], X_all.loc[i_hold, best_cols])
    thr, f = optimise_threshold(y_all[i_hold], p)
    print(f"drop sensor-NaN training rows = {str(drop_na):5s}  n_train={len(itr):,}  holdout best F1={f:.5f}")
    log_result(dict(stage=f"B_drop_nan_{drop_na}", holdout_f1_oracle_thr=f, n_features=len(best_cols)))

## 8. Stability and importance

In [ ]:
# ---- per-day stability of the chosen configuration ------------------------
i_pre = np.flatnonzero(dt_all < pd.Timestamp(HOLDOUT_START))
i_hold = np.flatnonzero(dt_all >= pd.Timestamp(HOLDOUT_START))
_, p_hold = fit_predict(X_all.loc[i_pre, best_cols], y_all[i_pre], X_all.loc[i_hold, best_cols])
thr, _ = optimise_threshold(y_all[i_hold], p_hold)
per_day = pd.DataFrame({
    "date": dt_all[i_hold].dt.date.values,
    "y": y_all[i_hold],
    "pred": (p_hold >= thr).astype(int),
})
g = per_day.groupby("date").apply(
    lambda s: pd.Series({"n": len(s), "f1": f1_score(s.y, s.pred),
                         "true_rate": s.y.mean(), "pred_rate": s.pred.mean()}))
print(g.round(4).to_string())
print(f"\nper-day F1 spread: {g.f1.std():.5f} (min {g.f1.min():.5f}, max {g.f1.max():.5f})")

imp = pd.Series(fit_predict(X_all.loc[i_pre, best_cols], y_all[i_pre],
                            X_all.loc[i_hold, best_cols])[0].get_feature_importance(),
                index=best_cols).sort_values(ascending=False)
print("\ntop features"); print(imp.head(20).round(3).to_string())
imp.to_csv(os.path.join(WORK, "feature_importance_temporal.csv"))

## 9. Light model refinement

In [ ]:
# ---- light refinement, only after the feature set is settled --------------
TUNE_CONFIGS = {
    "tune_0": dict(depth=6, learning_rate=0.08, iterations=600),
    "tune_1": dict(depth=8, learning_rate=0.06, iterations=900),
    "tune_2": dict(depth=6, learning_rate=0.05, iterations=1200),
}
for name, g_ in TUNE_CONFIGS.items():
    p = dict(CB_PARAMS); p.update(g_)
    print(f"\n=== {name} {g_} ===")
    run_temporal_cv(best_cols, name, params=p, verbose=False)

log_df = save_log("notebookB_metrics.csv")
with open(os.path.join(WORK, "notebookB_summary.json"), "w") as fh:
    json.dump({r["stage"]: {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                            for k, v in r.items() if k != "stage"} for r in METRIC_LOG}, fh, indent=2)
print(f"\nruntime so far: {(time.time()-t_notebook_start)/60:.1f} min")


## 10. Final models on all training data + candidate submissions
Files are written to `/kaggle/working`. Nothing is submitted.

In [ ]:
# ---- final models on ALL training data + safe submission helpers -----------
def fit_full_probs(cols, params=None, n_seeds=1):
    """Fit on all labelled rows and return test probabilities in X_test order."""
    t0 = time.time()
    probs = np.zeros(len(X_test), dtype=np.float64)
    base = dict(params or CB_PARAMS)
    for s in range(n_seeds):
        p = dict(base)
        p["random_seed"] = SEED + s
        m = CatBoostClassifier(**p)
        m.fit(X_all[cols], y_all)
        probs += m.predict_proba(X_test[cols])[:, 1]
    probs /= n_seeds
    assert np.isfinite(probs).all(), "non-finite test probabilities"
    print(f"  fitted {len(cols)} features x {n_seeds} seed(s) in {time.time()-t0:.0f}s")
    return probs


def write_submission(probs, threshold, tag):
    """Write one submission with hard guards against the previous NaN-threshold bug."""
    threshold = float(threshold)
    assert np.isfinite(threshold), f"{tag}: threshold is not finite: {threshold}"
    assert 0.0 < threshold < 1.0, f"{tag}: suspicious threshold {threshold}"
    pred = (np.asarray(probs) >= threshold).astype(int)
    pos_rate = float(pred.mean())
    assert set(np.unique(pred)).issubset({0, 1})
    # This task's labelled positive rate is ~0.37. A 0% or 100% prediction rate is
    # almost certainly a pipeline failure, so fail loudly instead of writing junk.
    assert 0.01 < pos_rate < 0.99, f"{tag}: degenerate predicted positive rate {pos_rate:.6f}"

    sub = pd.DataFrame({"ID": test_ids, TARGET: pred}).sort_values("ID").reset_index(drop=True)
    if sample is not None:
        assert len(sub) == len(sample), f"row count {len(sub)} != sample {len(sample)}"
        assert set(sub["ID"]) == set(sample["ID"]), "ID mismatch against sample_submission"
        sub = sub.set_index("ID").loc[sample["ID"]].reset_index()
    assert len(sub) == len(X_test)
    assert sub[TARGET].isin([0, 1]).all()

    path = os.path.join(WORK, f"submission_{tag}.csv")
    sub.to_csv(path, index=False)
    np.save(os.path.join(WORK, f"test_proba_{tag}.npy"), np.asarray(probs))
    print(f"{tag:30s} thr={threshold:.4f}  positive rate={pos_rate:.4f} -> {os.path.basename(path)}")
    return sub


def metric_row(stage):
    """Return the newest metric row for a named stage, requiring a usable honest threshold."""
    df = pd.DataFrame(METRIC_LOG)
    r = df[df["stage"].eq(stage)].copy()
    if "holdout_thr_honest" not in r.columns:
        raise KeyError(f"{stage}: no holdout_thr_honest column")
    r = r[np.isfinite(pd.to_numeric(r["holdout_thr_honest"], errors="coerce"))]
    if r.empty:
        raise ValueError(f"{stage}: no finite honestly-chosen threshold")
    return r.iloc[-1]


In [ ]:
# ---- candidate submissions (FIXED; nothing is submitted automatically) ----
final = pd.DataFrame(METRIC_LOG)

# Only REAL model configurations are eligible for final selection. Diagnostic
# rows such as ordering_check and B_drop_nan_* are deliberately absent here.
FINAL_CONFIGS = {
    "S3_neighbour": {
        "cols": STAGES["S3_neighbour"],
        "params": CB_PARAMS,
    },
    "S4_rolling": {
        "cols": best_cols,
        "params": CB_PARAMS,
    },
    "B_full": {
        "cols": best_cols,
        "params": CB_PARAMS,
    },
    "B_no_weather": {
        "cols": ablations["B_no_weather"],
        "params": CB_PARAMS,
    },
    "B_plus_calendar": {
        "cols": ablations["B_plus_calendar"],
        "params": CB_PARAMS,
    },
}
for tune_name, overrides in TUNE_CONFIGS.items():
    p = dict(CB_PARAMS); p.update(overrides)
    FINAL_CONFIGS[tune_name] = {"cols": best_cols, "params": p}

eligible = final[
    final["stage"].isin(FINAL_CONFIGS)
    & np.isfinite(pd.to_numeric(final.get("holdout_f1_honest_thr"), errors="coerce"))
    & np.isfinite(pd.to_numeric(final.get("holdout_thr_honest"), errors="coerce"))
].copy()
assert not eligible.empty, "No eligible model row has a finite honest holdout threshold"
best_row = eligible.loc[eligible["holdout_f1_honest_thr"].idxmax()]
best_stage = str(best_row["stage"])
best_thr = float(best_row["holdout_thr_honest"])
best_cfg = FINAL_CONFIGS[best_stage]

print("best REAL configuration by strict-holdout F1 with an honestly chosen threshold:")
show_cols = [c for c in ["stage", "n_features", "fwd_f1_mean", "fwd_f1_std",
                         "holdout_f1_honest_thr", "holdout_thr_honest"] if c in best_row.index]
print(best_row[show_cols].to_string())
assert np.isfinite(best_thr), "best threshold is NaN/inf"
print(f"selected final stage={best_stage}, threshold={best_thr:.6f}, features={len(best_cfg['cols'])}")

# 1) Neighbour-only candidate gets ITS OWN threshold (not the best model's threshold).
row_nb = metric_row("S3_neighbour")
thr_nb = float(row_nb["holdout_thr_honest"])
probs_nb = fit_full_probs(STAGES["S3_neighbour"], CB_PARAMS, n_seeds=1)
write_submission(probs_nb, thr_nb, "B_neighbour")

# 2) Full rolling candidate gets ITS OWN honest threshold; reuse the same
#    probabilities for the 0.5 safety/reference submission.
# Prefer the explicit B_full ablation row because it is evaluated after all stages.
full_stage = "B_full" if (final["stage"] == "B_full").any() else "S4_rolling"
row_full = metric_row(full_stage)
thr_full = float(row_full["holdout_thr_honest"])
probs_full = fit_full_probs(best_cols, CB_PARAMS, n_seeds=3)
write_submission(probs_full, thr_full, "B_full_holdoutthr")
write_submission(probs_full, 0.5, "B_full_thr050")

# 3) No-weather candidate: both validation schemes previously suggested this
#    may generalise better, so generate the EXACT feature set with ITS threshold.
row_nw = metric_row("B_no_weather")
thr_nw = float(row_nw["holdout_thr_honest"])
probs_nw = fit_full_probs(ablations["B_no_weather"], CB_PARAMS, n_seeds=3)
write_submission(probs_nw, thr_nw, "B_no_weather")

# 4) Exact best eligible strict-temporal model. Reuse already-computed probs
#    when possible; otherwise fit the matching feature set/params.
if best_stage == "S3_neighbour":
    probs_best = probs_nb
elif best_stage in ("S4_rolling", "B_full"):
    probs_best = probs_full
elif best_stage == "B_no_weather":
    probs_best = probs_nw
else:
    probs_best = fit_full_probs(best_cfg["cols"], best_cfg["params"], n_seeds=3)
write_submission(probs_best, best_thr, "B_best_strict")

# 5) Ordering-averaged version of the SAME selected best model. The previous
#    notebook could pair this with NaN because ordering_check was allowed to win;
#    now both the columns/params and threshold are guaranteed to match best_stage.
print("\nordering-averaged prediction (3 within-minute orderings) for", best_stage)
prob_by_id = np.zeros(len(test), dtype=np.float64)
id_ref = None
for s in (0, 1, 2):
    fl_s, feat_s, grp_s = build_features(order_seed=s)
    m_s = (fl_s["is_test"] == 0).values
    Xa = feat_s[m_s].reset_index(drop=True)
    ya = fl_s.loc[m_s, TARGET].astype(int).values
    Xt = feat_s[~m_s].reset_index(drop=True)
    ids = fl_s.loc[~m_s, "ID"].astype(int).values

    cols = best_cfg["cols"]
    params = dict(best_cfg["params"])
    params["random_seed"] = SEED  # vary ordering only, not model randomness
    mdl = CatBoostClassifier(**params)
    mdl.fit(Xa[cols], ya)
    p = mdl.predict_proba(Xt[cols])[:, 1]
    o = np.argsort(ids)
    prob_by_id += p[o]
    if id_ref is None:
        id_ref = ids[o]
    else:
        assert np.array_equal(id_ref, ids[o]), "test ID alignment changed across order seeds"
    print(f"  ordering seed {s} done")
    del fl_s, feat_s, Xa, Xt; gc.collect()
prob_by_id /= 3

# Convert ID-sorted averaged probabilities back to X_test/test_ids order before
# passing to the common writer, which performs the final sample-submission alignment.
prob_map = pd.Series(prob_by_id, index=id_ref)
probs_ordered = prob_map.loc[test_ids].values
write_submission(probs_ordered, best_thr, "B_order_averaged")

# Save the exact choice so there is no ambiguity when reviewing outputs.
selection = {
    "best_stage": best_stage,
    "best_threshold": best_thr,
    "n_features": int(len(best_cfg["cols"])),
    "feature_columns": list(best_cfg["cols"]),
    "note": "Diagnostic rows were excluded from model selection; every submission uses a finite threshold matched to its own configuration."
}
with open(os.path.join(WORK, "notebookB_final_selection.json"), "w") as fh:
    json.dump(selection, fh, indent=2)

print(f"\nfiles in {WORK}:")
for f_ in sorted(os.listdir(WORK)):
    print("  ", f_)
print(f"\ntotal runtime: {(time.time()-t_notebook_start)/60:.1f} min")
print("\nNothing has been submitted. Compare the candidates before choosing.")


## What to bring back

* `notebookB_metrics.csv` - forward-chaining folds, honest vs oracle thresholds, holdout detail
* `notebookB_final_selection.json` - exact eligible best model, threshold, and feature list
* the ordering-trap numbers from section 5 and the hint-3 verdict from section 6
* `submission_B_neighbour.csv` - neighbour model with its own honest threshold
* `submission_B_full_holdoutthr.csv` - full rolling model with its own honest threshold
* `submission_B_full_thr050.csv` - same full probabilities at threshold 0.5
* `submission_B_no_weather.csv` - no-weather ablation with its own honest threshold
* `submission_B_best_strict.csv` - exact best eligible strict-temporal configuration
* `submission_B_order_averaged.csv` - same best configuration averaged over three within-minute orderings

**Safety fix:** diagnostic rows can no longer be selected as final models, NaN thresholds are rejected, and degenerate 0%/100% positive-rate submissions fail loudly instead of being written silently.
